# 02 — Sectoral scaling spectrum and the urbanization crossover

Reproduces Sections 2.1–2.3 of *"Deepening before dissolution: non-monotonic escape
dynamics in the scaling exponents of county economies"*:
the sectoral spectrum of population-scaling exponents (primary sublinear, secondary
superlinear), the interaction model with its urbanization crossover $u^*$, the sigmoid
phase-transition fit of $\beta_{\mathrm{eff}}(u)$, the progressive land-control ladder,
and the replication of the crossover under two independent satellite urbanization
measures (nighttime lights and GAIA impervious surface).

In [1]:
import sys, os
sys.path.insert(0, '/home/yushentang/EconP-release')
os.chdir('/home/yushentang/EconP-release')

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd

from src.scaling import (
    run_scaling_regression,
    run_interaction_regression,
    run_urbanization_stratified_scaling,
    fit_sigmoid_crossover,
    run_land_variable_comparison,
)

panel = pd.read_parquet('data_clean/panel_scaling.parquet')
print(f"Analysis panel: {panel.shape[0]:,} county-year rows, "
      f"{panel['CountyCode'].nunique():,} counties, "
      f"years {panel['Year'].min()}-{panel['Year'].max()}")

Analysis panel: 19,392 county-year rows, 1,882 counties, years 2000-2023


## 1. Sectoral spectrum of scaling exponents (Section 2.1)

For each year we regress $\ln Y_{\mathrm{sector}}$ on $\ln N$ (population) separately for
primary, secondary and tertiary GDP, then average the yearly exponents. The paper reports a
stable spectrum — primary sublinear ($\bar\beta \approx 0.87$), tertiary near-linear
($\approx 0.96$), secondary superlinear ($\approx 1.08$) — with the ordering
$\beta_{\mathrm{pri}} < \beta_{\mathrm{ter}} < \beta_{\mathrm{sec}}$ holding in 23 of 24 years.

In [2]:
sectors = {"Primary": "RegGDP_Primary",
           "Secondary": "RegGDP_Secondary",
           "Tertiary": "RegGDP_Tertiary"}

rows = []
for year in sorted(panel["Year"].unique()):
    for name, col in sectors.items():
        r = run_scaling_regression(panel, y_col=col, x_cols=["PopNum"], year=year)
        rows.append({"Year": year, "Sector": name, "Beta": r.betas["PopNum"]})

yearly = pd.DataFrame(rows).pivot(index="Year", columns="Sector", values="Beta")
means = yearly.mean()

print(f"Yearly-mean beta, primary:   {means['Primary']:.3f}   (paper reports: 0.87)")
print(f"Yearly-mean beta, tertiary:  {means['Tertiary']:.3f}   (paper reports: 0.96)")
print(f"Yearly-mean beta, secondary: {means['Secondary']:.3f}   (paper reports: 1.08)")

ordering = (yearly["Primary"] < yearly["Tertiary"]) & (yearly["Tertiary"] < yearly["Secondary"])
print(f"Ordering pri < ter < sec held in {ordering.sum()} of {len(yearly)} years "
      f"(paper reports: 23 of 24)")

Yearly-mean beta, primary:   0.873   (paper reports: 0.87)
Yearly-mean beta, tertiary:  0.962   (paper reports: 0.96)
Yearly-mean beta, secondary: 1.080   (paper reports: 1.08)
Ordering pri < ter < sec held in 23 of 24 years (paper reports: 23 of 24)


## 2. Interaction model: urbanization activates superlinearity (Section 2.2)

The interaction specification lets the exponent depend linearly on the urbanization ratio
$u = Y_{\mathrm{nonagri}}/Y_{\mathrm{total}}$:

$$\ln Y = \alpha + (\beta_0 + \beta_1 u)\,\ln N + \varepsilon .$$

A positive $\beta_1$ means urbanization steepens the scaling; the effective exponent crosses
$\beta = 1$ at $u^* = (1-\beta_0)/\beta_1$. The paper reports strongly positive slopes for the
non-agricultural sectors ($\beta_1$: secondary 1.95, tertiary 1.48, non-agricultural 1.70) with
crossovers at $u^* \approx 0.75/0.80/0.79$, against an essentially flat primary sector
($\beta_1 = 0.05$, $p = 7\times10^{-5}$).

In [3]:
targets = [("Secondary", "RegGDP_Secondary", 1.95, 0.75),
           ("Tertiary", "RegGDP_Tertiary", 1.48, 0.80),
           ("NonAgri", "NonAgriGDP", 1.70, 0.79),
           ("Primary", "RegGDP_Primary", 0.05, None)]

for name, col, paper_b1, paper_ustar in targets:
    res = run_interaction_regression(panel, y_col=col)
    line = (f"{name:10s} beta1 = {res['beta1']:.3f} (p = {res['p_beta1']:.1e})"
            f"   paper reports: {paper_b1}")
    if name == "Primary":
        line += " with p = 7e-5; no meaningful crossover"
    else:
        line += f"; u* = {res['crossover_u']:.3f} (paper reports: {paper_ustar})"
    print(line)

Secondary  beta1 = 1.952 (p = 0.0e+00)   paper reports: 1.95; u* = 0.749 (paper reports: 0.75)
Tertiary   beta1 = 1.483 (p = 0.0e+00)   paper reports: 1.48; u* = 0.801 (paper reports: 0.8)
NonAgri    beta1 = 1.702 (p = 0.0e+00)   paper reports: 1.7; u* = 0.786 (paper reports: 0.79)
Primary    beta1 = 0.047 (p = 6.8e-05)   paper reports: 0.05 with p = 7e-5; no meaningful crossover


## 3. Sigmoid phase-transition fit of the effective exponent (Section 2.2)

Counties are stratified into 20 urbanization quantiles; a local exponent is estimated within
each bin and the profile $\beta_{\mathrm{eff}}(u)$ is fitted with a four-parameter sigmoid,
compared against a linear model by AIC. The paper reports a sharp crossover for tertiary GDP
($k = 23.2$, $u^* = 0.786$, $\Delta\mathrm{AIC} = -14.2$) and a near-step transition for
non-agricultural GDP ($k = 58.0$, $u^* = 0.739$); secondary GDP shows no improvement over the
linear model under the GDP-share measure (revisited with satellite measures in Section 5).

In [4]:
strat = run_urbanization_stratified_scaling(panel, n_quantiles=20)
tab = strat["table"]
group_stats = strat["group_stats"]
u_of_group = group_stats["mean_ratio"]  # bin centers (mean urbanization per quantile)

paper_vals = {"Tertiary": (23.2, 0.786, -14.2), "NonAgri": (58.0, 0.739, -7.2)}

for sector in ["Tertiary", "NonAgri"]:
    sub = tab[tab["Sector"] == sector]
    u = u_of_group.loc[sub["Group"]].values.astype(float)
    fit = fit_sigmoid_crossover(u, sub["Beta"].values, sub["SE"].values)
    sig, lin = fit["sigmoid"], fit["linear"]
    dAIC = sig["AIC"] - lin["AIC"]
    pk, pu, pd_ = paper_vals[sector]
    print(f"{sector:10s} k = {sig['k']:.1f} (paper reports: {pk}), "
          f"u* = {sig['u_star']:.3f} (paper reports: {pu}), "
          f"dAIC(sigmoid-linear) = {dAIC:.1f} (paper reports: {pd_})")

Tertiary   k = 23.2 (paper reports: 23.2), u* = 0.786 (paper reports: 0.786), dAIC(sigmoid-linear) = -14.2 (paper reports: -14.2)
NonAgri    k = 58.0 (paper reports: 58.0), u* = 0.739 (paper reports: 0.739), dAIC(sigmoid-linear) = -7.2 (paper reports: -7.2)


### Bootstrap confidence interval for the crossover

The province-block bootstrap of the tertiary crossover (500 draws, seed 42) is a heavy
computation and is **not** rerun here. We read the stored bootstrap draws written by
`scripts/run_core_statistics.py` and recompute the percentile interval. The paper reports
a 95% CI of $[0.721,\ 0.915]$ for $u^*_{\mathrm{ter}}$.

In [5]:
# Full run (about 30 s of bootstrap inside a longer script):
# python scripts/run_core_statistics.py
u_stars = np.load("results/tables/core/bootstrap_u_stars.npy")
lo, hi = np.percentile(u_stars, [2.5, 97.5])
print(f"Bootstrap draws: n = {len(u_stars)} (n_boot = 500, seed = 42)")
print(f"95% CI for u*_tertiary: [{lo:.3f}, {hi:.3f}]   (paper reports: [0.721, 0.915])")

Bootstrap draws: n = 500 (n_boot = 500, seed = 42)
95% CI for u*_tertiary: [0.721, 0.915]   (paper reports: [0.721, 0.915])


## 4. Dual diminishment: the land-control ladder (Section 2.3)

The sublinear primary exponent could reflect either crowding of a fixed natural input (land)
or an intrinsic absence of interaction-driven increasing returns. Progressive land controls
separate the channels: the paper reports that the population exponent of agricultural output
drops from $\hat\beta_N = 0.95$ (no control) to $0.87$ (cultivated area) to $0.63$ (total sown
area). As a placebo, the same control applied to non-agricultural GDP moves $\hat\beta_N$ only
from $1.03$ to $0.96$, with a land coefficient indistinguishable from zero.

In [6]:
r_nocontrol = run_scaling_regression(panel, y_col="GrossOutputValue",
                                     x_cols=["PopNum"], year="pooled")
print(f"beta_N, no land control:        {r_nocontrol.betas['PopNum']:.3f}"
      f"   (paper reports: 0.95)")

land = run_land_variable_comparison(panel)
pop_rows = land[(land["Specification"] == "Pop + Land") & (land["Variable"] == "PopNum")]
b_cult = pop_rows.loc[pop_rows["LandVar"] == "CultivateArea", "Beta"].iloc[0]
b_sown = pop_rows.loc[pop_rows["LandVar"] == "TotalSownArea", "Beta"].iloc[0]
print(f"beta_N | cultivated-area control: {b_cult:.3f}   (paper reports: 0.87)")
print(f"beta_N | total-sown-area control: {b_sown:.3f}   (paper reports: 0.63)")

# Placebo: land control on non-agricultural GDP
r_na = run_scaling_regression(panel, y_col="NonAgriGDP", x_cols=["PopNum"], year="pooled")
r_na_land = run_scaling_regression(panel, y_col="NonAgriGDP",
                                   x_cols=["PopNum", "TotalSownArea"], year="pooled")
print(f"Placebo (non-agri GDP): beta_N {r_na.betas['PopNum']:.3f} -> "
      f"{r_na_land.betas['PopNum']:.3f} with land coefficient "
      f"{r_na_land.betas['TotalSownArea']:.3f}   (paper reports: 1.03 -> 0.96, land ~ 0)")

beta_N, no land control:        0.953   (paper reports: 0.95)
beta_N | cultivated-area control: 0.872   (paper reports: 0.87)
beta_N | total-sown-area control: 0.631   (paper reports: 0.63)
Placebo (non-agri GDP): beta_N 1.032 -> 0.955 with land coefficient 0.051   (paper reports: 1.03 -> 0.96, land ~ 0)


## 5. Replication under independent satellite urbanization measures (Section 2.2)

Because $u = Y_{\mathrm{nonagri}}/Y_{\mathrm{total}}$ shares its numerator with the dependent
variables, the crossover analysis is replicated with a nighttime-lights measure
$u_{\mathrm{lit}}$ (fraction of county area lit above threshold) that has no algebraic overlap
with GDP. These analyses require the satellite rasters and are precomputed by
`scripts/run_crossover_nightlights.py`; we read the stored tables. The paper reports that
$\beta_{\mathrm{eff}}$ rises across the 20 quantiles in all three non-agricultural sectors,
the sigmoid beats the linear model with $\Delta\mathrm{AIC}$ between $-15$ and $-30$
(quantile-mapped scale), and the interaction slopes preserve their ordering
($\beta_1 = 1.15/1.08/1.11$ for secondary/tertiary/non-agri against $0.24$ for primary).

In [7]:
# Full run: python scripts/run_crossover_nightlights.py
ulit = pd.read_csv("results/tables/crossover_nightlights.csv")
qm = ulit[ulit["space"] == "qmapped"].copy()
qm["rising"] = qm["beta_hi"] > qm["beta_lo"]
print("Nighttime lights (u_lit), quantile-mapped scale:")
for _, row in qm.iterrows():
    print(f"  {row['sector']:10s} beta_eff rising: {bool(row['rising'])}, "
          f"dAIC(sigmoid-linear) = {row['dAIC_sig_minus_lin']:.1f}")
print(f"Rising in {int(qm['rising'].sum())}/3 sectors (paper reports: 3/3); "
      f"dAIC range [{qm['dAIC_sig_minus_lin'].min():.1f}, {qm['dAIC_sig_minus_lin'].max():.1f}] "
      f"(paper reports: between -30 and -15)")

inter = pd.read_csv("results/tables/interaction_nightlights.csv").set_index("y_col")
b1 = inter["beta1"]
print()
print(f"Interaction slopes under u_lit:")
print(f"  Secondary: {b1['RegGDP_Secondary']:.2f} (paper reports: 1.15)")
print(f"  Tertiary:  {b1['RegGDP_Tertiary']:.2f} (paper reports: 1.08)")
print(f"  NonAgri:   {b1['NonAgriGDP']:.2f} (paper reports: 1.11)")
print(f"  Primary:   {b1['RegGDP_Primary']:.2f} (paper reports: 0.24)")

Nighttime lights (u_lit), quantile-mapped scale:
  Tertiary   beta_eff rising: True, dAIC(sigmoid-linear) = -29.7
  Secondary  beta_eff rising: True, dAIC(sigmoid-linear) = -15.2
  NonAgri    beta_eff rising: True, dAIC(sigmoid-linear) = -20.7
Rising in 3/3 sectors (paper reports: 3/3); dAIC range [-29.7, -15.2] (paper reports: between -30 and -15)

Interaction slopes under u_lit:
  Secondary: 1.15 (paper reports: 1.15)
  Tertiary:  1.08 (paper reports: 1.08)
  NonAgri:   1.11 (paper reports: 1.11)
  Primary:   0.24 (paper reports: 0.24)


### Cross-validation with GAIA impervious-surface share

A second, physically distinct satellite measure — impervious-surface share
$u_{\mathrm{imp}}$ (GAIA, 30 m, 2000–2018) — cross-validates the nightlights replication on
its subsample. Precomputed by `scripts/run_crossover_impervious.py`. The paper reports:
$\beta_{\mathrm{eff}}$ rises in all three sectors, the sigmoid is preferred throughout
(quantile-mapped $\Delta\mathrm{AIC}$ between $-13$ and $-22$), the interaction ordering holds
($\beta_1 = 2.31/1.94/2.11$ against $0.56$ for primary), and the two satellite measures agree
with each other (Spearman $\rho = 0.83$) while each is only moderately related to the GDP
share ($\rho = 0.39$ and $0.59$).

In [8]:
# Full run: python scripts/run_crossover_impervious.py
gaia = pd.read_csv("results/tables/crossover_impervious.csv")

cross = gaia[(gaia["measure"] == "u_imp") & (gaia["space"] == "qmapped")
             & (gaia["analysis"] == "crossover")]
print("GAIA impervious surface (u_imp), quantile-mapped scale:")
for _, row in cross.iterrows():
    print(f"  {row['sector']:10s} beta_eff rising: {row['rising']}, "
          f"dAIC(sigmoid-linear) = {row['dAIC_sig_minus_lin']:.1f}")
n_rising = int(cross['rising'].astype(bool).sum())
print(f"Rising in {n_rising}/3 sectors (paper reports: 3/3); "
      f"dAIC range [{cross['dAIC_sig_minus_lin'].min():.1f}, "
      f"{cross['dAIC_sig_minus_lin'].max():.1f}] (paper reports: between -22 and -13)")

inter_g = gaia[(gaia["measure"] == "u_imp") & (gaia["analysis"] == "interaction")]
inter_g = inter_g.set_index("sector")["int_beta1"]
print()
print("Interaction slopes under u_imp:")
print(f"  Secondary: {inter_g['Secondary']:.2f} (paper reports: 2.31)")
print(f"  Tertiary:  {inter_g['Tertiary']:.2f} (paper reports: 1.94)")
print(f"  NonAgri:   {inter_g['NonAgri']:.2f} (paper reports: 2.11)")
print(f"  Primary:   {inter_g['Primary']:.2f} (paper reports: 0.56)")

corr = pd.read_csv("results/tables/urbanisation_measure_correlations.csv").set_index("pair")
print()
print("Agreement between urbanization measures (Spearman rho):")
print(f"  u_imp ~ u_lit: {corr.loc['u_imp~u_lit', 'spearman']:.2f} (paper reports: 0.83)")
print(f"  u_imp ~ u_GDP: {corr.loc['u_imp~u_GDP', 'spearman']:.2f} (paper reports: 0.39)")
print(f"  u_lit ~ u_GDP: {corr.loc['u_lit~u_GDP', 'spearman']:.2f} (paper reports: 0.59)")

GAIA impervious surface (u_imp), quantile-mapped scale:
  Tertiary   beta_eff rising: True, dAIC(sigmoid-linear) = -21.7
  Secondary  beta_eff rising: True, dAIC(sigmoid-linear) = -12.8
  NonAgri    beta_eff rising: True, dAIC(sigmoid-linear) = -17.0
Rising in 3/3 sectors (paper reports: 3/3); dAIC range [-21.7, -12.8] (paper reports: between -22 and -13)

Interaction slopes under u_imp:
  Secondary: 2.31 (paper reports: 2.31)
  Tertiary:  1.94 (paper reports: 1.94)
  NonAgri:   2.11 (paper reports: 2.11)
  Primary:   0.56 (paper reports: 0.56)

Agreement between urbanization measures (Spearman rho):
  u_imp ~ u_lit: 0.83 (paper reports: 0.83)
  u_imp ~ u_GDP: 0.39 (paper reports: 0.39)
  u_lit ~ u_GDP: 0.59 (paper reports: 0.59)


## Summary

The sectoral spectrum (sublinear primary, superlinear secondary), the urbanization-activated
crossover of the effective exponent, the land-control decomposition of agricultural
sublinearity, and the satellite replications of the crossover all reproduce the values
reported in Sections 2.1–2.3 of the paper from the pinned analysis panel and the committed
result tables.